<div align="center">

## **DS108 - TIỀN XỬ LÝ VÀ XÂY DỰNG BỘ DỮ LIỆU**

## **ĐỒ ÁN - THU THẬP & TIỀN XỬ LÝ DỮ LIỆU TÀI CHÍNH DOANH NGHIỆP CHO DỰ ĐOÁN LỢI NHUẬN**
</div>

**GVHD:** TS.Nguyễn Gia Tuấn Anh - CN: Trần Quốc Khánh

**Sinh viên thực hiện:**

| Họ và tên | MSSV |
|---|---|
| Võ Hạnh Nguyên | 24521218 |
| Phan Thị Ánh Nguyệt | 24521221 |
  
**Nguồn dữ liệu:** Alpha Vantage Financial API  
**Notebook:** `01_data_collection.ipynb` — Thu thập dữ liệu thô

---

## **Mục tiêu**
Notebook này thực hiện thu thập **Báo cáo Kết quả Kinh doanh** (Income Statement) theo quý của **23 doanh nghiệp niêm yết** trên thị trường chứng khoán Mỹ thông qua Alpha Vantage API, bao gồm:
- Gọi API và xử lý các trường hợp lỗi
- Chuẩn hoá cấu trúc dữ liệu đầu ra
- Lưu dữ liệu thô theo từng doanh nghiệp và dạng tổng hợp

**Biến mục tiêu (target):** `netIncome` — Lợi nhuận ròng sau thuế

---

### ***Pipeline:***
1. Cài đặt thư viện
2. Cấu hình toàn cục
3. Định nghĩa các cột thu thập
4. Hàm thu thập dữ liệu từ API
5. Hàm crawl toàn bộ danh sách
6. Chạy pipeline & lưu kết quả
7. Kiểm tra dữ liệu đầu ra

---
## **1. Cài đặt thư viện**

In [1]:
# ── Khai báo các thư viện cần thiết ──────────────────────────────────────────────
import requests     # Gửi HTTP request đến Alpha Vantage API
import pandas as pd # Xử lý và lưu trữ dữ liệu dạng bảng
import os           # Thao tác hệ thống tệp
import time         # Điều chỉnh tốc độ gọi API (sleep)
import logging      # Ghi log quá trình thu thập
from datetime import datetime   # Đặt tên file theo thời gian
from pathlib import Path        # Quản lý đường dẫn thư mục
from getpass import getpass       # Che API key

print("✅ Import thư viện thành công.")

✅ Import thư viện thành công.


---
## **2. Cấu hình toàn cục** 
Tập trung toàn bộ tham số cấu hình ở một cell duy nhất để dễ chỉnh sửa khi cần thay đổi API key, danh sách mã cổ phiếu hoặc tần suất thu thập.

In [2]:
# ── Thông tin API ──────────────────────────────────────────────────────────
API_KEY  = getpass("Nhập API key của bạn:")
BASE_URL = "https://www.alphavantage.co/query"

# ── Danh sách mã cổ phiếu cần thu thập ────────────────────────────────────
# Gồm 23 doanh nghiệp blue-chip đa ngành trên sàn NYSE / NASDAQ
# Ngành: Công nghệ · Tài chính · Bán lẻ · Y tế · Năng lượng · Viễn thông
SYMBOLS = [
    'AAPL', 'AMZN', 'AVGO', 'BABA', 'BRK-B',   # Apple, Amazon, Broadcom, Alibaba, Berkshire
    'COST', 'GOOG', 'HD',   'JNJ',  'JPM',       # Costco, Google, Home Depot, J&J, JPMorgan
    'LLY',  'MA',   'META', 'MSFT', 'NFLX',      # Eli Lilly, Mastercard, Meta, Microsoft, Netflix
    'NVDA', 'ORCL', 'PLTR', 'TSLA', 'TSM',       # Nvidia, Oracle, Palantir, Tesla, TSMC
    'V',    'WMT',  'XOM',                        # Visa, Walmart, ExxonMobil
]

# ── Cấu hình thu thập ─────────────────────────────────────────────────────
PERIOD = "quarterly"          # Tần suất: "quarterly" (quý) hoặc "annual" (năm)
                              # Chọn "quarterly" để có nhiều điểm dữ liệu hơn cho ML

# Delay giữa các request để không vượt quá giới hạn free tier
# Free tier: 5 req/phút → mỗi 60s/5 = 12s; đặt 13s để có biên an toàn
REQUEST_DELAY_SECONDS = 13

# ── Thư mục lưu dữ liệu ───────────────────────────────────────────────────
RAW_DIR  = Path("data/raw")   # Dữ liệu thô chưa qua xử lý
LOG_DIR  = Path("logs")       # File log quá trình crawl

RAW_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Cấu hình hoàn tất.")
print(f"   Số mã cổ phiếu : {len(SYMBOLS)}")
print(f"   Tần suất        : {PERIOD}")
print(f"   Delay mỗi request: {REQUEST_DELAY_SECONDS}s")
print(f"   Thư mục đầu ra  : {RAW_DIR}")

✅ Cấu hình hoàn tất.
   Số mã cổ phiếu : 23
   Tần suất        : quarterly
   Delay mỗi request: 13s
   Thư mục đầu ra  : data\raw


---
## **3. Cấu hình hệ thống ghi log**

Ghi log song song ra file và console để dễ theo dõi tiến độ khi chạy lâu (pipeline 23 mã × 13s ≈ ~5 phút).

In [3]:
# Cấu hình logging: ghi đồng thời ra file log và màn hình console
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "crawl.log", encoding="utf-8"),  # Lưu vào file
        logging.StreamHandler(),                                         # Hiện trên màn hình
    ],
)
logger = logging.getLogger(__name__)
logger.info("Khởi tạo hệ thống logging thành công.")

2026-05-21 00:21:00,065 [INFO] Khởi tạo hệ thống logging thành công.


---
## **4. Định nghĩa các cột thu thập**

Alpha Vantage trả về nhiều trường hơn mức cần thiết. Bảng ánh xạ dưới đây xác định chính xác 26 chỉ tiêu tài chính cần giữ lại, được tổ chức theo cấu trúc của báo cáo kết quả kinh doanh từ trên xuống dưới.

In [4]:
# Ánh xạ: tên cột từ API Alpha Vantage → tên chuẩn trong dataset
# Cấu trúc theo thứ tự từ trên xuống của Income Statement:
#   Revenue → Gross Profit → Operating Income → Non-operating → EBT → Net Income

INCOME_COLUMNS = {
    # ── Thông tin cơ bản ───────────────────────────────────────────────────
    "fiscalDateEnding":                  "fiscalDateEnding",       # Ngày kết thúc kỳ báo cáo
    "reportedCurrency":                  "reportedCurrency",       # Đơn vị tiền tệ (thường là USD)

    # ── Doanh thu & Chi phí hàng bán (Revenue Layer) ───────────────────────
    "totalRevenue":                      "totalRevenue",           # Tổng doanh thu thuần
    "costOfRevenue":                     "costOfRevenue",          # Giá vốn hàng bán (COGS)
    "costofGoodsAndServicesSold":        "costofGoodsAndServicesSold", # COGS chi tiết (hàng hoá + DV)
    "grossProfit":                       "grossProfit",            # Lợi nhuận gộp = Revenue - COGS

    # ── Chi phí hoạt động (Operating Expenses Layer) ───────────────────────
    "sellingGeneralAndAdministrative":   "sellingGeneralAndAdministrative", # Chi phí bán hàng & QLDN
    "researchAndDevelopment":            "researchAndDevelopment", # Chi phí nghiên cứu & phát triển
    "operatingExpenses":                 "operatingExpenses",      # Tổng chi phí hoạt động
    "operatingIncome":                   "operatingIncome",        # Lợi nhuận từ hoạt động KD

    # ── Thu nhập / Chi phí ngoài hoạt động (Non-operating Layer) ──────────
    "investmentIncomeNet":               "investmentIncomeNet",    # Thu nhập ròng từ đầu tư
    "netInterestIncome":                 "netInterestIncome",      # Thu nhập lãi vay ròng
    "interestIncome":                    "interestIncome",         # Thu nhập từ lãi
    "interestExpense":                   "interestExpense",        # Chi phí lãi vay
    "nonInterestIncome":                 "nonInterestIncome",      # Thu nhập ngoài lãi vay
    "otherNonOperatingIncome":           "otherNonOperatingIncome",# Thu nhập phi hoạt động khác

    # ── Khấu hao (Depreciation) ────────────────────────────────────────────
    "depreciation":                      "depreciation",           # Khấu hao tài sản cố định
    "depreciationAndAmortization":       "depreciationAndAmortization", # Khấu hao & phân bổ

    # ── Lợi nhuận trước & sau thuế (Bottom Line) ───────────────────────────
    "incomeBeforeTax":                   "incomeBeforeTax",        # Lợi nhuận trước thuế (EBT)
    "incomeTaxExpense":                  "incomeTaxExpense",       # Chi phí thuế thu nhập
    "interestAndDebtExpense":            "interestAndDebtExpense", # Tổng chi phí lãi & nợ
    "netIncomeFromContinuingOperations": "netIncomeFromContinuingOperations", # LNST từ HĐ liên tục
    "comprehensiveIncomeNetOfTax":       "comprehensiveIncomeNetOfTax",       # Thu nhập toàn diện

    # ── Các chỉ số tổng hợp ────────────────────────────────────────────────
    "ebit":                              "ebit",    # Lợi nhuận trước lãi vay & thuế
    "ebitda":                            "ebitda",  # EBIT + Khấu hao
    "netIncome":                         "netIncome",  # ★ BIẾN MỤC TIÊU — Lợi nhuận ròng
}

print(f"✅ Định nghĩa {len(INCOME_COLUMNS)} cột cần thu thập.")
print(f"   Biến mục tiêu (target): netIncome")

✅ Định nghĩa 26 cột cần thu thập.
   Biến mục tiêu (target): netIncome


---
## **5. Hàm thu thập dữ liệu từ API**

Hàm `fetch_income_statement()` thực hiện một lần gọi API cho một mã cổ phiếu, xử lý đầy đủ các tình huống lỗi có thể xảy ra với Alpha Vantage.

In [5]:
def fetch_income_statement(symbol: str, period: str = "quarterly") -> pd.DataFrame | None:
    """
    Gọi Alpha Vantage API để lấy Income Statement của một mã cổ phiếu.

    Tham số:
        symbol  (str) : Mã cổ phiếu (ví dụ: "AAPL", "MSFT")
        period  (str) : Tần suất báo cáo — "quarterly" hoặc "annual"

    Trả về:
        pd.DataFrame  : Dữ liệu thu thập được, đã chuẩn hoá cột
        None          : Nếu xảy ra lỗi hoặc không có dữ liệu
    """

    # Tham số gửi đến API
    params = {
        "function": "INCOME_STATEMENT",
        "symbol":   symbol,
        "apikey":   API_KEY,
    }

    # ── Bước 1: Gửi HTTP request ───────────────────────────────────────────
    try:
        logger.info(f"[{symbol}] Đang gọi API ({period})...")
        response = requests.get(BASE_URL, params=params, timeout=30)
        response.raise_for_status()   # Ném lỗi nếu HTTP status không phải 2xx
        data = response.json()
    except requests.exceptions.Timeout:
        logger.error(f"[{symbol}] Lỗi: Request timeout (>30s).")
        return None
    except requests.exceptions.ConnectionError:
        logger.error(f"[{symbol}] Lỗi: Mất kết nối mạng.")
        return None
    except requests.exceptions.RequestException as e:
        logger.error(f"[{symbol}] Lỗi request không xác định: {e}")
        return None

    # ── Bước 2: Kiểm tra phản hồi từ phía API ─────────────────────────────
    if "Note" in data:
        # Xảy ra khi vượt quá giới hạn 5 request/phút (free tier)
        logger.warning(f"[{symbol}] Đã chạm giới hạn rate limit: {data['Note']}")
        return None

    if "Error Message" in data:
        # Mã cổ phiếu không hợp lệ hoặc không được hỗ trợ
        logger.error(f"[{symbol}] Lỗi từ API: {data['Error Message']}")
        return None

    if "Information" in data:
        # Thông báo yêu cầu nâng cấp lên gói Premium
        logger.warning(f"[{symbol}] Cần tài khoản Premium: {data['Information']}")
        return None

    # ── Bước 3: Trích xuất danh sách báo cáo ──────────────────────────────
    key     = "quarterlyReports" if period == "quarterly" else "annualReports"
    reports = data.get(key, [])

    if not reports:
        logger.warning(f"[{symbol}] Không có dữ liệu {period}.")
        return None

    # ── Bước 4: Chuẩn hoá DataFrame ───────────────────────────────────────
    df = pd.DataFrame(reports)

    # Chỉ giữ các cột đã định nghĩa trong INCOME_COLUMNS
    # (bỏ qua cột nếu API không trả về — tránh KeyError)
    available_cols = [c for c in INCOME_COLUMNS.keys() if c in df.columns]
    df = df[available_cols].copy()

    # Thêm cột định danh công ty và đổi tên cột sang tên chuẩn
    df.insert(0, "symbol", symbol)
    df.rename(columns=INCOME_COLUMNS, inplace=True)

    logger.info(f"[{symbol}] Thu thập được {len(df)} bản ghi ({period}).")
    return df


print("✅ Đã định nghĩa hàm fetch_income_statement().")

✅ Đã định nghĩa hàm fetch_income_statement().


---
## **6. Hàm crawl toàn bộ danh sách**

Hàm `crawl_all()` điều phối vòng lặp qua tất cả 23 mã cổ phiếu, áp dụng delay giữa các request, lưu file từng công ty và gộp thành file tổng hợp cuối cùng.

In [6]:
def crawl_all(symbols: list[str], period: str = "quarterly") -> pd.DataFrame:
    """
    Crawl toàn bộ danh sách mã cổ phiếu, lưu file raw từng công ty
    và gộp thành một file tổng hợp.

    Chiến lược lưu file kép:
      - Từng công ty  → data/raw/{SYMBOL}_income_{period}.csv
        (Đảm bảo không mất dữ liệu nếu pipeline bị ngắt giữa chừng)
      - Tổng hợp      → data/raw/Data_DS108_raw.csv
        (File đầu vào cho notebook tiền xử lý tiếp theo)

    Tham số:
        symbols (list) : Danh sách mã cổ phiếu
        period  (str)  : "quarterly" hoặc "annual"

    Trả về:
        pd.DataFrame   : Dataset tổng hợp tất cả công ty
    """
    all_frames = []   # Danh sách DataFrame của từng công ty thành công
    failed     = []   # Danh sách mã bị lỗi để báo cáo cuối

    logger.info("=" * 60)
    logger.info("BẮT ĐẦU THU THẬP DỮ LIỆU BÁO CÁO KẾT QUẢ KINH DOANH")
    logger.info(f"Tổng số mã cổ phiếu  : {len(symbols)}")
    logger.info(f"Tần suất báo cáo     : {period}")
    logger.info(f"Delay giữa request  : {REQUEST_DELAY_SECONDS}s")
    logger.info("=" * 60)

    for idx, symbol in enumerate(symbols, start=1):
        logger.info(f"── Đang xử lý ({idx}/{len(symbols)}): {symbol} ──")

        # Gọi hàm thu thập cho một mã cổ phiếu
        df = fetch_income_statement(symbol, period)

        if df is not None and not df.empty:
            # Lưu dữ liệu thô của từng công ty riêng lẻ
            raw_path = RAW_DIR / f"{symbol}_income_{period}.csv"
            df.to_csv(raw_path, index=False, encoding="utf-8-sig")
            logger.info(f"[{symbol}] Đã lưu → {raw_path}")
            all_frames.append(df)
        else:
            # Ghi nhận mã bị lỗi, tiếp tục với mã tiếp theo
            failed.append(symbol)
            logger.warning(f"[{symbol}] Bỏ qua do không có dữ liệu.")

        # Áp dụng delay để tuân thủ giới hạn rate limit của free tier
        # Không cần delay sau mã cuối cùng
        if idx < len(symbols):
            logger.info(f"Chờ {REQUEST_DELAY_SECONDS}s trước request tiếp theo...")
            time.sleep(REQUEST_DELAY_SECONDS)

    # ── Gộp và lưu file tổng hợp ──────────────────────────────────────────
    if all_frames:
        combined  = pd.concat(all_frames, ignore_index=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_path  = RAW_DIR / f"Data_DS108_raw.csv"
        combined.to_csv(out_path, index=False, encoding="utf-8-sig")

        logger.info("=" * 60)
        logger.info(f"✅ HOÀN TẤT! Tổng {len(combined):,} bản ghi từ {len(all_frames)} công ty.")
        logger.info(f"   File tổng hợp → {out_path}")
    else:
        combined = pd.DataFrame()
        logger.error("❌ Không thu thập được dữ liệu từ bất kỳ công ty nào!")

    # Báo cáo các mã bị lỗi (nếu có)
    if failed:
        logger.warning(f"Các mã bị lỗi ({len(failed)}): {failed}")
    else:
        logger.info("Tất cả mã cổ phiếu đều thu thập thành công.")

    return combined


print("✅ Đã định nghĩa hàm crawl_all().")

✅ Đã định nghĩa hàm crawl_all().


---
## **7. Chạy pipeline thu thập dữ liệu**

In [7]:
logger.info("=" * 60)
logger.info("BẮT ĐẦU THU THẬP DỮ LIỆU BÁO CÁO KẾT QUẢ TÀI CHÍNH")
logger.info(f"Số mã cổ phiếu: {len(SYMBOLS)}")
logger.info(f"API Key: {'***' + API_KEY[-4:] if len(API_KEY) > 4 else '(chưa cấu hình)'}")
logger.info("=" * 60)

df_raw = crawl_all(SYMBOLS, period="quarterly")

2026-05-21 00:21:00,103 [INFO] ============================================================
2026-05-21 00:21:00,104 [INFO] BẮT ĐẦU THU THẬP DỮ LIỆU BÁO CÁO KẾT QUẢ TÀI CHÍNH
2026-05-21 00:21:00,104 [INFO] Số mã cổ phiếu: 23
2026-05-21 00:21:00,105 [INFO] API Key: ***77UQ
2026-05-21 00:21:00,105 [INFO] ============================================================
2026-05-21 00:21:00,106 [INFO] ============================================================
2026-05-21 00:21:00,107 [INFO] BẮT ĐẦU THU THẬP DỮ LIỆU BÁO CÁO KẾT QUẢ KINH DOANH
2026-05-21 00:21:00,107 [INFO] Tổng số mã cổ phiếu  : 23
2026-05-21 00:21:00,108 [INFO] Tần suất báo cáo     : quarterly
2026-05-21 00:21:00,108 [INFO] Delay giữa request  : 13s
2026-05-21 00:21:00,109 [INFO] ============================================================
2026-05-21 00:21:00,109 [INFO] ── Đang xử lý (1/23): AAPL ──
2026-05-21 00:21:00,109 [INFO] [AAPL] Đang gọi API (quarterly)...
2026-05-21 00:21:02,644 [INFO] [AAPL] Thu thập được 81 bản ghi (